# Notebook 01: Data Ingestion Pipeline

## Overview

This notebook walks through the complete ingestion pipeline:

```
NICE PDF  -->  Clean Markdown  -->  Structure-Aware Chunks  -->  Embeddings  -->  Chroma
```

**What you will see:**
- How raw PDF pages are extracted and cleaned
- How the text is split into token-aware chunks
- How chunk metadata is structured for retrieval
- How embeddings are created and stored in Chroma

**Source files:**
- `src/rag_app/ingestion/pdf_loader.py` - PDF loading and markdown cleaning
- `src/rag_app/ingestion/chunker.py` - Structure-aware recursive chunking
- `src/rag_app/ingestion/indexer.py` - Embedding and Chroma indexing

## 1. Setup

Add the project root to `sys.path` so we can import from `src.rag_app`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_app import config

## 2. Load the PDF

We use `pymupdf4llm` to extract each page as markdown, preserving headings.
The loader tracks the current heading across pages so each chunk knows its section.

In [ ]:
from src.rag_app.ingestion.pdf_loader import load_pdf

pages = load_pdf()
print(f"Loaded {len(pages)} pages from {config.PDF_PATH.name}")
print(f"\nFirst page sample (page {pages[0]['page_number']}):")
print(f"  Section: {pages[0]['section_title']}")
print(f"  Text length: {len(pages[0]['text'])} chars")
print(f"  Preview: {pages[0]['text'][:200]}...")

## 3. See the Cleaning in Action

The `clean_markdown()` function removes PDF conversion noise:
- HTML tags (`<br>`, `<u>`, `<strong>`)
- Markdown formatting (`*`, `#`)
- Extra whitespace and semicolons used as line breaks in tables

In [ ]:
from src.rag_app.ingestion.pdf_loader import clean_markdown

# Show cleaning on a sample with common PDF noise
dirty_text = """
<strong>1.3.1</strong> Offer quantitative faecal immunochemical testing (FIT)<br>
to guide referral for suspected colorectal cancer in adults with:
- an abdominal mass and/or
- a change in bowel habit persisting >= 6 weeks
"""

clean_text = clean_markdown(dirty_text)
print("BEFORE cleaning:")
print(dirty_text)
print("\nAFTER cleaning:")
print(clean_text)

## 4. Load the Embedding Model

We use `intfloat/multilingual-e5-base` which supports both Arabic and English.
The tokenizer is critical because chunk sizes are measured in E5 tokens, not words.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    config.EMBEDDING_MODEL,
    local_files_only=config.EMBEDDING_LOCAL_FILES_ONLY,
)
print(f"Model: {config.EMBEDDING_MODEL}")
print(f"Max sequence length: {model.get_sentence_embedding_dimension()} dimensions")

# Show how the tokenizer counts tokens
sample = "Offer quantitative faecal immunochemical testing to guide referral"
tokens = model.tokenizer.encode(sample, add_special_tokens=False)
print(f"\nSample text: '{sample}'")
print(f"Token count: {len(tokens)}")
print(f"CHUNK_SIZE setting: {config.CHUNK_SIZE} tokens")

## 5. Create Chunks

The chunking pipeline:
1. **Filter pages** - only useful guideline content (pages 5-28, 30-48)
2. **Split at recommendation boundaries** - each `1.x.x` number starts a new chunk
3. **Merge cross-page continuations** - tables and recommendations that span pages
4. **Split oversized chunks** - keep tables readable while fitting E5's 512-token limit
5. **Assign content types** - recommendation, glossary, rationale, etc.

In [ ]:
from src.rag_app.ingestion.chunker import (
    chunk_pages,
    content_type_for_page,
    token_count,
)
from src.rag_app.ingestion.pdf_loader import load_ng12_colorectal_chunks

# Create chunks from the main guideline
chunks = chunk_pages(pages, model.tokenizer)

# Add supplementary NICE NG12 chunks (colorectal symptoms/referral)
supplementary = load_ng12_colorectal_chunks(model.tokenizer)
chunks.extend(supplementary)

print(f"Total chunks: {len(chunks)}")
print(f"  NG151 guideline chunks: {len(chunks) - len(supplementary)}")
print(f"  NG12 supplementary chunks: {len(supplementary)}")

## 6. Examine Chunk Structure

Each chunk carries metadata that supports retrieval and citation:
- `chunk_id` - unique identifier (e.g., `ng151-p10-c3`)
- `document_name` - which NICE guideline
- `page_number` - for citation
- `section_title` - guideline section
- `content_type` - recommendation, glossary, rationale, etc.
- `text` - the full text including metadata prefix (used for embedding)
- `content` - just the guideline text (returned in search results)

In [ ]:
# Show a sample recommendation chunk
sample_chunk = chunks[0]
print("Sample chunk metadata:")
for key in ["chunk_id", "document_name", "page_number", "section_title", "content_type"]:
    print(f"  {key}: {sample_chunk[key]}")

print(f"\nContent ({len(sample_chunk['content'])} chars):")
print(sample_chunk["content"][:300])

print(f"\nText for embedding ({len(sample_chunk['text'])} chars):")
print(sample_chunk["text"][:300])

## 7. Analyze Chunk Size Distribution

Good chunks should be:
- **Long enough** to contain meaningful context (not just headings)
- **Short enough** to fit within the embedding model's limit (512 tokens)
- **Consistent** in size for predictable retrieval quality

In [ ]:
# Analyze token counts across all chunks
token_counts = [token_count(c["content"], model.tokenizer) for c in chunks]

print("Chunk token count statistics:")
print(f"  Min: {min(token_counts)} tokens")
print(f"  Max: {max(token_counts)} tokens")
print(f"  Mean: {sum(token_counts) / len(token_counts):.0f} tokens")
print(f"  CHUNK_SIZE setting: {config.CHUNK_SIZE} tokens")
print(f"  E5 model limit: 512 tokens")

# Distribution by content type
from collections import Counter
type_counts = Counter(c["content_type"] for c in chunks)
print(f"\nChunks by content type:")
for content_type, count in type_counts.most_common():
    print(f"  {content_type}: {count}")

## 8. Build the Chroma Index

Chroma stores the embeddings with cosine similarity search.
We pre-compute all embeddings and store them directly (no embedding function).

In [ ]:
import chromadb

# Build the index
from src.rag_app.ingestion.indexer import build_index

build_index(chunks, model)

# Verify the collection
client = chromadb.PersistentClient(path=str(config.CHROMA_PATH))
collection = client.get_collection(config.COLLECTION_NAME)
print(f"Chroma collection '{config.COLLECTION_NAME}' created successfully")
print(f"Total vectors: {collection.count()}")

## 9. Verify: Quick Test Search

A quick sanity check to confirm the index works.

In [ ]:
# Quick test: embed a query and search
query = "What follow-up is recommended after surgery?"
query_vector = model.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
    convert_to_numpy=True,
)

results = collection.query(
    query_embeddings=query_vector.tolist(),
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: '{query}'")
print(f"\nTop 3 results:")
for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    score = 1 - float(dist)
    print(f"\n  #{i} (score: {score:.4f})")
    print(f"     Page: {meta['page_number']}, Section: {meta['section_title']}")
    print(f"     Preview: {doc[:120]}...")

## Summary

The ingestion pipeline:

| Step | Input | Output |
|------|-------|--------|
| PDF loading | NICE NG151 PDF | 48 page records with headings |
| Markdown cleaning | Raw PDF text | Clean text without HTML/formatting noise |
| Structure-aware chunking | Pages | ~95 citation-ready chunks |
| Supplementary extraction | NICE NG12 PDF | 5 additional colorectal chunks |
| Embedding | Chunk text | 512-dim vectors (multilingual-e5-base) |
| Chroma indexing | Vectors + metadata | Persistent cosine HNSW index |

**Key design decisions:**
- Chunk size of 450 tokens (under E5's 512 limit, leaves room for metadata)
- Split at NICE recommendation boundaries (`1.x.x` numbers)
- Cross-page continuation merging for tables and long recommendations
- Content type filtering during retrieval (only search `recommendation` chunks)